# Warehouse agent over A2A (Week 6)

This notebook is the **client** half of the Agent2Agent (A2A) lesson: it uses `a2a-sdk` to discover the remote agent card, open a streaming connection, send a `Message`, and print reply artifacts.

**Before you run cells:** start the warehouse A2A server from the repo root with secrets resolved, e.g. `make run-a2a-warehouse-agent`, and set `BASE_URL` to match that process (`HOST`/`PORT`, default port `10000` in `appy.py`). If `.env` uses `op://` references, use `make jupyter-lab` and attach this notebook to that server so `OPENAI_API_KEY` is not left as a literal `op://` string.

## Imports

In [ ]:
import uuid

import httpx
from a2a.client import A2ACardResolver, A2AClient, ClientFactory
from a2a.types import (
    AgentCard,
    Message,
    MessageSendParams,
    Part,
    Role,
    SendMessageRequest,
    TextPart,
)

In [ ]:
BASE_URL = "http://localhost:10000"
PUBLIC_AGENT_CARD_PATH = "/.well-known/agent.json"

In [ ]:
async with httpx.AsyncClient() as httpx_client:
    # Initialize A2ACardResolver
    resolver = A2ACardResolver(
        httpx_client=httpx_client,
        base_url=BASE_URL,
    )

    try:
        print(
            f"Fetching public agent card from: {BASE_URL}{PUBLIC_AGENT_CARD_PATH}"
        )
        _public_card = await resolver.get_agent_card()
        print("Fetched public agent card")
        print(_public_card.model_dump_json(indent=2))

        final_agent_card_to_use = _public_card

    except Exception as e:
        print(f"Error fetching public agent card: {e}")
        raise RuntimeError("Failed to fetch public agent card")

In [ ]:
timeout_config = httpx.Timeout(
    connect=10.0,    # Connection timeout
    read=100.0,      # Read timeout (important for long-running operations)
    write=10.0,      # Write timeout
    pool=10.0        # Pool timeout
)

client = await ClientFactory.connect(
    agent=BASE_URL,
    resolver_http_kwargs={"timeout": timeout_config}
)

print("A2AClient initialized")

message_id = str(uuid.uuid4())
message_payload = Message(
    role=Role.user,
    messageId=message_id,
    parts=[Part(root=TextPart(text="Hello, how are you?"))],
)

print("Sending message")
final_response = None
async for response in client.send_message(message_payload):
    final_response = response

print("Response:")
for artifact in final_response[0].artifacts:
    for part in artifact.parts:
        print(part.root.text)